# Lab 03 · Seleccionar y filtrar

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crapdev/sparkLab/blob/main/labs/03-seleccionar-filtrar/lab.ipynb)

**Objetivo:** Quedarte con las columnas y las filas que te interesan de un DataFrame: select, col, alias, filter y las condiciones combinadas con &, | y ~.

**Tiempo estimado:** 35 minutos

## Antes de empezar

1. Ve a **Archivo → Guardar una copia en Drive**. Así trabajas sobre tu copia y no pierdes nada.
2. Ejecuta las celdas **en orden**, de arriba abajo (`Shift + Enter`).
3. Donde veas `...` te toca escribir código. Las celdas de test te dicen si está bien:
   ✅ es correcto y ❌ viene con una pista.
4. Si te atascas, vuelve a la teoría del módulo antes de mirar la solución.

## Preparación

Esta celda instala la versión de PySpark con la que se probó el laboratorio y comprueba
que hay una versión de Java compatible. En Colab tarda alrededor de un minuto la primera vez.

In [ ]:
import importlib.metadata
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.2.0"
MIN_JAVA_VERSION = 17


def installed_pyspark_version():
    try:
        return importlib.metadata.version("pyspark")
    except importlib.metadata.PackageNotFoundError:
        return None


def java_major_version():
    java = shutil.which("java")
    if java is None:
        return 0
    output = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
    match = re.search(r'version "(\d+)', output)
    return int(match.group(1)) if match else 0


if installed_pyspark_version() != PYSPARK_VERSION:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )

if java_major_version() < MIN_JAVA_VERSION:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Spark 4 necesita Java 17 o superior. Instálalo y repite esta celda.")
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print(f"PySpark {PYSPARK_VERSION} y Java {java_major_version()} listos.")

La siguiente celda define `check`, la función que usan los tests. No hace falta que la
modifiques: ejecútala y sigue.

In [ ]:
RESULTS = {}


def check(name, condition, hint):
    """Run one test: `condition` is a function that must return True."""
    try:
        passed = bool(condition())
        error = None
    except Exception as exc:  # A broken answer is a failed test, not a crash.
        passed, error = False, exc
    RESULTS[name] = passed
    if passed:
        print(f"✅ {name}")
        return
    detail = f" ({type(error).__name__}: {error})" if error else ""
    print(f"❌ {name}{detail}")
    print(f"   Pista: {hint}")
    assert passed, f"{name} · {hint}"

## Sesión de Spark

Esta celda crea la `SparkSession` del laboratorio e importa las funciones como `F`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("local[*]").appName("lab-03-seleccionar-filtrar").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo.")

## Datos

Esta celda descarga `bigmart_sales.csv` la primera vez y los deja en la carpeta `data/`.
Usa `dataset_path("nombre")` para obtener la ruta de un archivo.

In [ ]:
import os
import urllib.request

DATA_BASE_URLS = [
    "https://raw.githubusercontent.com/crapdev/sparkLab/main/public/data",
]
DATA_DIR = os.environ.get("PYSPARK_LABS_DATA_DIR", "data")
DATASETS = ["bigmart_sales.csv"]


def dataset_path(name):
    """Return the local path of a dataset, downloading it the first time."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    os.makedirs(DATA_DIR, exist_ok=True)
    for base_url in DATA_BASE_URLS:
        try:
            urllib.request.urlretrieve(f"{base_url}/{name}", path)
            return path
        except OSError:
            continue  # Try the next mirror.
    raise RuntimeError(f"No se pudo descargar {name}. Revisa tu conexión y repite la celda.")


for dataset in DATASETS:
    print("Listo:", dataset_path(dataset))

In [ ]:
sales = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(dataset_path("bigmart_sales.csv"))
)

print("Filas:", sales.count(), "· Columnas:", len(sales.columns))
print(sales.columns)

## Tarea 1 · Selecciona tres columnas

`select` devuelve un DataFrame nuevo solo con las columnas que le pides. La forma más
corta es pasar los nombres como strings.

Crea `item_basics` con exactamente estas tres columnas, en este orden:
`Item_Identifier`, `Item_Weight` e `Item_Fat_Content`.

In [ ]:
item_basics = ...

In [ ]:
check(
    "Tarea 1 · tres columnas en orden",
    lambda: item_basics.columns == ["Item_Identifier", "Item_Weight", "Item_Fat_Content"],
    'Usa sales.select("Item_Identifier", "Item_Weight", "Item_Fat_Content").',
)
check(
    "Tarea 1 · siguen las 8523 filas",
    lambda: item_basics.count() == 8523,
    "select recorta columnas, no filas: tienen que quedar todas.",
)

Mira el resultado. Fíjate en que `select` no toca las filas: hay las mismas que antes,
pero más estrechas.

In [ ]:
item_basics.show(3)

## Tarea 2 · Renombra con alias

Un string solo puede nombrar una columna. Para hacer algo más con ella, como
renombrarla, hace falta un objeto columna: `F.col("nombre")`. A ese objeto se le puede
pedir `.alias("otro_nombre")`.

Crea `prices` con dos columnas: `Item_Identifier` renombrada a `Item_ID` e `Item_MRP`
renombrada a `Price`.

In [ ]:
prices = ...

In [ ]:
check(
    "Tarea 2 · columnas Item_ID y Price",
    lambda: prices.columns == ["Item_ID", "Price"],
    'Cada columna va como F.col("nombre_original").alias("nombre_nuevo").',
)
check(
    "Tarea 2 · Price contiene el precio",
    lambda: prices.first()["Price"] == 249.8092,
    "Price tiene que salir de Item_MRP, no de otra columna.",
)
check(
    "Tarea 2 · siguen las 8523 filas",
    lambda: prices.count() == 8523,
    "Solo hay que seleccionar y renombrar, sin filtrar filas.",
)

Un detalle importante: el alias cambia el nombre **en el DataFrame nuevo**. En `sales`
la columna sigue llamándose `Item_MRP`, y en `prices` ya no existe con ese nombre:

In [ ]:
prices.show(3)
print("Columnas de sales:  ", sales.columns[:2], "...")
print("Columnas de prices: ", prices.columns)

## Tarea 3 · Un filtro sencillo

`filter` devuelve las filas que cumplen una condición. La condición se escribe comparando
una columna con un valor: `F.col("columna") == valor`. `where` hace exactamente lo mismo.

Crea `regular` con los productos cuyo `Item_Fat_Content` es exactamente `"Regular"`.

In [ ]:
regular = ...

In [ ]:
check(
    "Tarea 3 · 2889 filas",
    lambda: regular.count() == 2889,
    'La condición es F.col("Item_Fat_Content") == "Regular", respetando mayúsculas.',
)
check(
    "Tarea 3 · todas las filas son Regular",
    lambda: regular.filter(F.col("Item_Fat_Content") != "Regular").count() == 0,
    "Se han colado filas con otro valor. Revisa la comparación.",
)
check(
    "Tarea 3 · conserva las 12 columnas",
    lambda: len(regular.columns) == 12,
    "filter recorta filas, no columnas: tienen que quedar las 12.",
)

Si escribes mal el valor, Spark no avisa: simplemente no encuentra ninguna fila.
Compara `"Regular"` con `"regular"`:

In [ ]:
print('"Regular":', sales.filter(F.col("Item_Fat_Content") == "Regular").count())
print('"regular":', sales.filter(F.col("Item_Fat_Content") == "regular").count())

Antes de filtrar por un valor de texto, conviene mirar qué valores hay de verdad:

In [ ]:
sales.select("Item_Fat_Content").distinct().show()

## Tarea 4 · Dos condiciones con `&`

Para exigir dos condiciones a la vez se usa `&`, y **cada condición va entre
paréntesis**: `(cond1) & (cond2)`. En PySpark no sirven `and`, `or` ni `not`: se usan
`&`, `|` y `~`.

Crea `light_drinks` con los productos de tipo `"Soft Drinks"` que pesan menos de 10
(columna `Item_Weight`).

In [ ]:
light_drinks = ...

In [ ]:
check(
    "Tarea 4 · 166 filas",
    lambda: light_drinks.count() == 166,
    "Las dos condiciones van entre paréntesis y unidas con &.",
)
check(
    "Tarea 4 · todas son Soft Drinks",
    lambda: light_drinks.filter(F.col("Item_Type") != "Soft Drinks").count() == 0,
    'La primera condición es F.col("Item_Type") == "Soft Drinks".',
)
check(
    "Tarea 4 · todas pesan menos de 10",
    lambda: light_drinks.filter(~(F.col("Item_Weight") < 10)).count() == 0,
    'La segunda condición es F.col("Item_Weight") < 10.',
)

¿Qué pasa si usas `and` en vez de `&`? Python intenta convertir la primera condición en
`True` o `False`, y una columna de Spark no es ni lo uno ni lo otro. Esta celda provoca
el error a propósito y muestra el mensaje:

In [ ]:
try:
    sales.filter((F.col("Item_Type") == "Soft Drinks") and (F.col("Item_Weight") < 10))
except Exception as error:
    print(type(error).__name__)
    print(error)

Olvidar los paréntesis produce **el mismo error**, porque `&` se evalúa antes que `==` y
`<`. Spark acaba viendo `"Soft Drinks" & F.col("Item_Weight")`, que no tiene sentido:

In [ ]:
try:
    sales.filter(F.col("Item_Type") == "Soft Drinks" & F.col("Item_Weight") < 10)
except Exception as error:
    print(type(error).__name__)
    print(error)

## Tarea 5 · Nulos y listas de valores

Dos condiciones nuevas:

- `F.col("columna").isNull()` es verdadera cuando el valor falta. Comparar con `== None`
  **no funciona**: siempre devuelve cero filas.
- `F.col("columna").isin("a", "b", "c")` es verdadera cuando el valor es uno de la lista.

Crea `unknown_size` con las filas que no tienen `Outlet_Size` **y** cuyo
`Outlet_Location_Type` es `"Tier 1"` o `"Tier 2"`.

In [ ]:
unknown_size = ...

In [ ]:
check(
    "Tarea 5 · 1855 filas",
    lambda: unknown_size.count() == 1855,
    'Combina F.col("Outlet_Size").isNull() y F.col("Outlet_Location_Type").isin("Tier 1", "Tier 2") con &.',
)
check(
    "Tarea 5 · Outlet_Size es nulo en todas",
    lambda: unknown_size.filter(F.col("Outlet_Size").isNotNull()).count() == 0,
    "Para detectar nulos usa .isNull(), no == None.",
)
check(
    "Tarea 5 · no hay tiendas de Tier 3",
    lambda: unknown_size.filter(F.col("Outlet_Location_Type") == "Tier 3").count() == 0,
    'Revisa la lista de isin: solo "Tier 1" y "Tier 2".',
)

Comprueba por qué `== None` no sirve. Las dos expresiones parecen pedir lo mismo, pero
solo una funciona:

In [ ]:
print("isNull(): ", sales.filter(F.col("Outlet_Size").isNull()).count())
print("== None:  ", sales.filter(F.col("Outlet_Size") == None).count())  # Wrong on purpose.

## Tarea 6 · Combina `|` y `~`

`|` es el OR y `~` es el NOT. Se combinan con `&` siguiendo la misma regla: cada
condición entre paréntesis.

Crea `fresh_outside_tier3` con los productos de tipo `"Dairy"` **o** `"Meat"` cuyo
`Outlet_Location_Type` **no** es `"Tier 3"`. Usa `|` para el "o" y `~` para el "no".

In [ ]:
fresh_outside_tier3 = ...

In [ ]:
check(
    "Tarea 6 · 667 filas",
    lambda: fresh_outside_tier3.count() == 667,
    "El OR de los dos tipos va entre paréntesis, y después el & con la negación.",
)
check(
    "Tarea 6 · solo Dairy y Meat",
    lambda: fresh_outside_tier3.filter(~F.col("Item_Type").isin("Dairy", "Meat")).count() == 0,
    'Las dos condiciones del OR son F.col("Item_Type") == "Dairy" y F.col("Item_Type") == "Meat".',
)
check(
    "Tarea 6 · ninguna tienda es de Tier 3",
    lambda: fresh_outside_tier3.filter(F.col("Outlet_Location_Type") == "Tier 3").count() == 0,
    'La negación es ~(F.col("Outlet_Location_Type") == "Tier 3"), con sus paréntesis.',
)

Si te ha salido, prueba a escribir lo mismo con `isin` en lugar del OR y con `!=` en
lugar del NOT. El resultado tiene que ser idéntico: hay varias formas de decir lo mismo.

In [ ]:
same_thing = sales.filter(F.col("Item_Type").isin("Dairy", "Meat") & (F.col("Outlet_Location_Type") != "Tier 3"))
print("Con | y ~:     ", fresh_outside_tier3.count())
print("Con isin y !=: ", same_thing.count())

## Para terminar · los nulos no se dejan comparar

`Item_Weight` tiene 1463 valores nulos. Si separas las filas en "pesan menos de 10" y
"no pesan menos de 10", esperarías que las dos partes sumen 8523. No lo hacen:

In [ ]:
light = sales.filter(F.col("Item_Weight") < 10).count()
not_light = sales.filter(~(F.col("Item_Weight") < 10)).count()
missing = sales.filter(F.col("Item_Weight").isNull()).count()

print("Item_Weight < 10:    ", light)
print("~(Item_Weight < 10): ", not_light)
print("Suma:                ", light + not_light, "de", sales.count())
print("Item_Weight nulo:    ", missing)

Una comparación con un nulo no es ni verdadera ni falsa, así que `filter` descarta esas
filas en los dos casos. Si quieres conservarlas, tienes que pedirlo: `(F.col("Item_Weight") < 10) | F.col("Item_Weight").isNull()`.

## Resumen

Ejecuta esta celda al final. Si todos los tests pasan, vuelve al sitio y marca el
laboratorio como completado.

In [ ]:
EXPECTED_TESTS = 17
passed_tests = sum(RESULTS.values())

print(f"{passed_tests} de {EXPECTED_TESTS} tests superados")
for test_name, passed in RESULTS.items():
    print("  ✅" if passed else "  ❌", test_name)

assert passed_tests == EXPECTED_TESTS, "Todavía quedan tareas por resolver."
print("\n🎉 ¡Laboratorio completado! Vuelve al sitio y márcalo como hecho.")